<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day08_practice3_%EC%A6%9D%EA%B0%95_%EC%8B%A4%EC%B8%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 증강 효과 실측
# "데이터가 적을 때 증강이 도움이 된다" 를 WDB로 기록하여 실측 - 그런데 결과가 데이터에 따라 갈린다.

In [4]:
!pip install -U wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: wandb
    Found existing installation: wandb 0.28.1
    Uninstalling wandb-0.28.1:
      Successfully uninstalled wandb-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.44.0 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.3

In [5]:
import os

os.environ["WANDB_MODE"] = "online"     # wandb를 '온라인'으로 (웹 서버에 기록)

import wandb

print("WANDB_MODE =", os.environ.get("WANDB_MODE"))

WANDB_MODE = online


In [6]:
# 로그인
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

import wandb
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: gimm00999 (gimm00999-kwu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset # 데이터 일부만 뽑는 Subset
from torchvision import datasets, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("장치:", device)

N_TRAIN = 5000
EPOCHS = 12


장치: cuda


In [8]:
DATASET = "cifar10"
N_TRAIN = 5000

In [10]:
# [셀 1] 데이터 준비 - 증강 유/무 두 열의 학습셋
def get_datasets(root='./data'):
  if DATASET in ("auto", "cifar10"):
    aug32 = [transforms.RandomHorizontalFlip(), transforms.RandomCrop(32, padding=4)]
    tr_aug = datasets.CIFAR10(root, train=True, download=True, transform=transforms.Compose(aug32 + [transforms.ToTensor()])) # 증강 학습셋
    tr_no = datasets.CIFAR10(root, train=True, download=True, transform=transforms.ToTensor()) # 무증강 학습셋
    te = datasets.CIFAR10(root, train=False, download=True, transform=transforms.ToTensor()) # 테스트셋(증강 없음)
    return "CIFAR-10", tr_aug, tr_no, te

name, train_aug, train_no, test_set = get_datasets()
C = train_no[0][0].shape[0] # 채널수 : 첫 샘플 (이미지, 라벨) -> (C, H, W) -> C
print(f"데이터: {name} | 학습 제한 {N_TRAIN}장 (원래 {len(train_no)}장)")

# 같은 5000장을 두 실험이 공유 -> 공정 비교 (증강 유무만 차이나게)
idx = torch.randperm(len(train_no), generator=torch.Generator().manual_seed(0))[:N_TRAIN].tolist()
test_loader = DataLoader(test_set, batch_size=512, shuffle=False) # 테스트 로더 (고정)


100%|██████████| 170M/170M [41:54<00:00, 67.8kB/s]


데이터: CIFAR-10 | 학습 제한 5000장 (원래 50000장)


In [11]:
# [셀 2] 공통 모델 - 평가 도구
def make_cnn(in_ch, img_size):
  feat = img_size // 4
  return nn.Sequential(
      nn.Conv2d(in_ch, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
      nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
      nn.Flatten(), nn.Linear(64 * feat * feat, 256), nn.ReLU(),
  )

def accuracy(model, loader):
  model.eval()
  c = t = 0
  with torch.no_grad():
    for x, y in loader:
      x, y = x.to(device), y.to(device)
      c += (model(x).argmax(1) == y).sum().item()
      t += len(y)
  return c / t # 정확도 (맞은수/전체수)

In [12]:
# [셀 3] 두 run - 증강 없음 vs 있음 (WDB 기록)
results = {}

for use_aug in [False, True]: # 증강 없음 -> 있음 순서로 두 번
  run_name = "with_aug" if use_aug else "no_aug"
  run = wandb.init(
      project="dl_day08_augmentation",
      name=run_name,
      config={"aug": use_aug, "n_train": N_TRAIN, "epochs": EPOCHS, "dataset": name, "speed": 42}
  )

  dataset = train_aug if use_aug else train_no

  train_loader = DataLoader(
      Subset(dataset, idx),
      batch_size=128,
      shuffle=True,
      generator=torch.Generator().manual_seed(42)
  )

  img_size = dataset[0][0].shape[-1]

  torch.manual_seed(42)
  model = make_cnn(C, img_size).to(device)
  loss_fn = nn.CrossEntropyLoss()
  opt = torch.optim.Adam(model.parameters(), lr=0.001)

  for epoch in range(EPOCHS):
    model.train()

    for x, y in train_loader:
      x, y = x.to(device), y.to(device)
      loss = loss_fn(model(x), y)

      opt.zero_grad()
      loss.backward()
      opt.step()

    tr_acc = accuracy(model, train_loader)
    te_acc = accuracy(model, test_loader)

    wandb.log({"train_acc": tr_acc, "test_acc": te_acc})

  results[run_name] = (tr_acc, te_acc)

  wandb.summary["final_train_acc"] = tr_acc
  wandb.summary["final_test_acc"] = te_acc

  wandb.finish()

  print(
      f"[{run_name:7s}] train {tr_acc:.3f} | "
      f"test {te_acc:.3f} | "
      f"격차 {tr_acc - te_acc:.3f}"
  )

test_acc,▁▂▄▆▆▆▇██▇▇█
train_acc,▁▂▃▅▆▅▆▇▇▇▇█
final_test_acc,0.4329
final_train_acc,0.4826
test_acc,0.4329
train_acc,0.4826


[no_aug ] train 0.483 | test 0.433 | 격차 0.050


test_acc,▁▃▄▆▆▆▇▇▇▇██
train_acc,▁▃▄▆▆▅▇▇▇▇██
final_test_acc,0.3828
final_train_acc,0.3696
test_acc,0.3828
train_acc,0.3696


[with_aug] train 0.370 | test 0.383 | 격차 -0.013
